# Junk Food Multi-class Classification with CLIP (zero-shot, Colab / Local)

Classifies every frame into exactly one **ad type** (`tip`) of `tip-multiclass-dataset.zip` with **zero-shot OpenCLIP
ViT-B-16** (`laion2b_s34b_b88k`, the same model as `CLIP-binary-classification.ipynb`). CLIP weights are never updated.

## Flow

Run the notebook top to bottom.

1. **Setup / data / split** - identical to the other multi-class notebooks, so the test images are the same. CLIP is
   zero-shot: only validation (prompt selection) and test are encoded.
2. **Prompts** - every class has candidate English concepts written from its `tip_label` description, each ensembled
   over five templates.
3. **Scoring** - the score of a class is the softmax over classes of the image-text cosine similarity (scaled by CLIP's
   logit scale); the prediction is the argmax.
4. **Selection on validation** - configurations "concept r" (every class uses its r-th concept) and "ensemble" (mean
   of all concepts per class) are ranked by validation macro-F1; rank 1 is the primary configuration.
5. **Test evaluation** - every configuration is reported on test in validation order; only rank 1 is the primary result.

Classes such as "supermarket not promoting food" vs "supermarket promoting food" differ in subtle content, so expect
zero-shot performance to vary a lot across classes.

## Environment Setup (Colab or Local)

Run the setup cell below **first**. It auto-detects Google Colab and, when running there, installs missing
dependencies, mounts Google Drive and extracts the dataset zip onto the local Colab disk (much faster than reading
thousands of JPEGs over Drive). Pretrained weights are downloaded fresh each session.

**Colab checklist**
1. `Runtime > Change runtime type > GPU` (T4 is enough; A100/L4 are faster).
2. Upload `tip-multiclass-dataset.zip` (from `unhealthy-dataset-generator`; it contains `tip_multiclass_dataset.csv` and `images/`) to Drive.
3. Add a Colab Secret (key icon in the left sidebar) named `MULTICLASS_DATASET_ZIP_PATH` with the path of that `.zip`, either absolute (`/content/drive/MyDrive/datasets/tip-multiclass-dataset.zip`) or relative to `/content/drive` (`MyDrive/datasets/tip-multiclass-dataset.zip`), enable notebook access, then run all cells.

Locally the dataset is read from `./dataset` (the extracted zip; override with the `JUNK_FOOD_DATASET_DIR` environment variable).

In [ ]:
# =====================================================================
# ENVIRONMENT SETUP - runs on Google Colab and locally.
#
# Colab:  installs missing packages, mounts Google Drive and extracts the
#         dataset zip to the local disk.
# Local:  no-op besides resolving ./dataset .
# =====================================================================
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))

CSV_NAME = "tip_multiclass_dataset.csv"
IMAGES_DIR_NAME = "images"
LOCAL_EXTRACT_DIR = Path("/content/multiclass_dataset")

# Local dataset location (used when not on Colab). Override with JUNK_FOOD_DATASET_DIR.
LOCAL_DATASET_DIR = next((p / "dataset" for p in (Path.cwd(), Path.cwd().parent) if (p / "dataset").exists()),
                         Path.cwd() / "dataset")


def _get_dataset_zip_path():
    """Colab Secret MULTICLASS_DATASET_ZIP_PATH, falling back to the environment variable of the same name."""
    if IN_COLAB:
        try:
            from google.colab import userdata

            value = userdata.get("MULTICLASS_DATASET_ZIP_PATH")
            if value:
                return value.strip()
        except Exception:  # secret missing or notebook access not granted
            pass
    return os.environ.get("MULTICLASS_DATASET_ZIP_PATH", "").strip() or None


def _find_dataset_root(base):
    """Return the folder holding tip_multiclass_dataset.csv and images/ (zips may add a wrapping folder)."""
    for current_dir, subdirs, file_names in os.walk(base):
        if CSV_NAME in file_names and IMAGES_DIR_NAME in subdirs:
            return Path(current_dir)
    return Path(base)


def _pip_install(*packages):
    missing = []
    for pkg, module in packages:
        try:
            __import__(module)
        except ImportError:
            missing.append(pkg)
    if missing:
        print(f"Installing: {', '.join(missing)}")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)


if IN_COLAB:
    print("Google Colab detected.")
    _pip_install(("open_clip_torch", "open_clip"))

    DATASET_ZIP_PATH = _get_dataset_zip_path()
    if not DATASET_ZIP_PATH:
        raise ValueError(
            "MULTICLASS_DATASET_ZIP_PATH is not set. Add a Colab Secret named MULTICLASS_DATASET_ZIP_PATH "
            "(key icon in the left sidebar, enable notebook access) pointing to tip-multiclass-dataset.zip "
            "in Google Drive, e.g. MyDrive/datasets/tip-multiclass-dataset.zip."
        )

    from google.colab import drive

    drive.mount("/content/drive")
    drive_source = (Path(DATASET_ZIP_PATH) if DATASET_ZIP_PATH.startswith("/content/")
                    else Path("/content/drive") / DATASET_ZIP_PATH.lstrip("/"))
    if not drive_source.exists():
        raise FileNotFoundError(f"Nothing found at {drive_source}. Check MULTICLASS_DATASET_ZIP_PATH.")

    # Always start from a clean folder so a previous/partial extraction can't leave stale files behind.
    if LOCAL_EXTRACT_DIR.exists():
        shutil.rmtree(LOCAL_EXTRACT_DIR)
    if drive_source.suffix.lower() == ".zip":
        print(f"Extracting {drive_source.name} ({drive_source.stat().st_size / 1e9:.2f} GB) to {LOCAL_EXTRACT_DIR}...")
        # Copy the archive off Drive first: one sequential read is far faster than unzipping over the mount.
        staged_zip = Path("/content") / drive_source.name
        shutil.copyfile(drive_source, staged_zip)
        with zipfile.ZipFile(staged_zip) as zf:
            zf.extractall(LOCAL_EXTRACT_DIR)
        staged_zip.unlink()
    else:
        print(f"Copying dataset folder from Drive to {LOCAL_EXTRACT_DIR} (slow; a .zip is much faster)...")
        shutil.copytree(drive_source, LOCAL_EXTRACT_DIR)
    DATASET_DIR = _find_dataset_root(LOCAL_EXTRACT_DIR)

    # Colab VMs have spare CPU cores for image decoding.
    DATALOADER_WORKERS = 2
else:
    print("Local environment detected.")
    DATASET_DIR = _find_dataset_root(LOCAL_DATASET_DIR) if LOCAL_DATASET_DIR.exists() else LOCAL_DATASET_DIR
    DATALOADER_WORKERS = 0

DATASET_DIR = Path(os.environ.get("JUNK_FOOD_DATASET_DIR", DATASET_DIR)).expanduser()

# --- Validate the dataset layout before anything expensive runs ---------
_csv = DATASET_DIR / CSV_NAME
_images = DATASET_DIR / IMAGES_DIR_NAME
_problems = [str(p) for p in (_csv, _images) if not p.exists()]
if _problems:
    raise FileNotFoundError(
        f"Dataset not found at {DATASET_DIR}. Missing: {', '.join(_problems)}.\n"
        "Expected layout:\n"
        f"  <dataset dir>/{CSV_NAME}\n"
        f"  <dataset dir>/{IMAGES_DIR_NAME}/*.jpg"
    )

print(f"Dataset dir: {DATASET_DIR}")
print(f"  images: {sum(1 for _ in _images.glob('*.jpg'))}")
print(f"  dataloader workers: {DATALOADER_WORKERS}")

In [ ]:
import json
import platform
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from PIL import Image
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
    precision_score,
    recall_score,
)
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# DATASET_DIR comes from the setup cell (extracted zip on Colab, ./dataset locally).
DATASET_ROOT = Path(DATASET_DIR)
CSV_PATH = DATASET_ROOT / CSV_NAME
IMAGES_DIR = DATASET_ROOT / IMAGES_DIR_NAME
IMG_EXTS = (".jpg", ".jpeg", ".png")

In [ ]:
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    props = torch.cuda.get_device_properties(0)
    vram_gb = props.total_memory / 1e9
    print(f"GPU: {props.name} ({vram_gb:.1f} GB)")
    EMBEDDING_BATCH_SIZE = 16 if vram_gb < 6 else (24 if vram_gb < 10 else 32)
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    print("Using Apple Metal Performance Shaders (MPS)")
    EMBEDDING_BATCH_SIZE = 8
else:
    DEVICE = torch.device("cpu")
    EMBEDDING_BATCH_SIZE = 8
    print("GPU not available, using CPU")
    if globals().get("IN_COLAB", False):
        print("On Colab, enable a GPU via Runtime > Change runtime type > T4 GPU "
              "and re-run from the setup cell; CPU is impractically slow.")

## Data and group-aware split

`tip_multiclass_dataset.csv` has one row per image: `image_name`, `tip` (class id), `tip_label` (class description) and
`group`. Each frame has exactly **one** class (multi-class, not multi-label). A group is an ad creative (`ad_XXXX`) or a
non-ad source video (`non_ad_<video>`); its frames are near-duplicates, so **whole groups** are assigned to
train/val/test (target 70/15/15 of the frames of each class, seed 42) and no group appears in two splits. A few
creatives contain frames of two classes (e.g. a supermarket ad that shows food only in some frames); the split works on
per-group class counts, so those groups are handled too.

Classes with fewer than 3 groups cannot appear in all splits: 1 group -> train only, 2 groups -> train + test (so the
class is still evaluated). The split summary prints a `NOTE` for these classes; their val/test metrics are very noisy.

Set `PILOT = True` for a fast run on a group-aware ~20% subset.

In [ ]:
# =====================================================================
# PILOT MODE: fast end-to-end pipeline check on a group-aware subset.
# =====================================================================
PILOT = False
PILOT_FRACTION = 0.20

# Short English names for the tip ids (descriptions come from the CSV).
SHORT_CLASS_NAMES = {
    0: "non_ad",
    2: "food_brand",
    3: "food_by_nonfood_co",
    5: "supermarket_food",
    6: "supermarket_nonfood",
    7: "restaurant_food",
    8: "restaurant_nonfood",
}

print("Loading CSV...")
raw_df = pd.read_csv(CSV_PATH, dtype=str, encoding="utf-8-sig")
missing_cols = {"image_name", "tip", "tip_label", "group"} - set(raw_df.columns)
if missing_cols:
    raise ValueError(f"{CSV_NAME} is missing columns: {sorted(missing_cols)}")
raw_df = raw_df.apply(lambda col: col.str.strip())

if raw_df["image_name"].duplicated().any():
    raise ValueError(f"Duplicated image_name rows: {raw_df.loc[raw_df['image_name'].duplicated(), 'image_name'].tolist()[:10]}")
raw_df["tip"] = pd.to_numeric(raw_df["tip"], errors="coerce")
if raw_df["tip"].isna().any():
    raise ValueError(f"Non-numeric tip values in {int(raw_df['tip'].isna().sum())} rows")
raw_df["tip"] = raw_df["tip"].astype(int)
empty_group = raw_df["group"].isna() | (raw_df["group"] == "")
if empty_group.any():
    raise ValueError(f"{int(empty_group.sum())} rows have no group, e.g. {raw_df.loc[empty_group, 'image_name'].tolist()[:10]}")

image_lookup = {p.name: p for p in IMAGES_DIR.iterdir() if p.suffix.lower() in IMG_EXTS}
missing_images = sorted(set(raw_df["image_name"]) - set(image_lookup))
if missing_images:
    raise FileNotFoundError(f"{len(missing_images)} CSV images are missing from images/, e.g. {missing_images[:10]}")
extra_images = sorted(set(image_lookup) - set(raw_df["image_name"]))
if extra_images:
    print(f"WARNING: {len(extra_images)} images in images/ are not in the CSV and are ignored, e.g. {extra_images[:5]}")

CLASS_IDS = sorted(raw_df["tip"].unique().tolist())
CLASS_DESCRIPTIONS = raw_df.groupby("tip")["tip_label"].first().to_dict()
CLASS_NAMES = [SHORT_CLASS_NAMES.get(t, f"tip_{t}") for t in CLASS_IDS]
NUM_CLASSES = len(CLASS_IDS)
TIP_TO_INDEX = {t: i for i, t in enumerate(CLASS_IDS)}

all_df = pd.DataFrame({
    "image_name": raw_df["image_name"],
    "image_path": raw_df["image_name"].map(lambda n: str(image_lookup[n])),
    "tip": raw_df["tip"],
    "tip_label": raw_df["tip_label"],
    "group": raw_df["group"],
})
all_df["y"] = all_df["tip"].map(TIP_TO_INDEX)
all_df["class_name"] = all_df["y"].map(lambda i: CLASS_NAMES[i])

print(f"\n{len(all_df)} images, {all_df['group'].nunique()} groups, {NUM_CLASSES} classes:")
print(pd.DataFrame({
    "tip": CLASS_IDS,
    "name": CLASS_NAMES,
    "images": [int((all_df["tip"] == t).sum()) for t in CLASS_IDS],
    "groups": [all_df.loc[all_df["tip"] == t, "group"].nunique() for t in CLASS_IDS],
    "description": [CLASS_DESCRIPTIONS[t] for t in CLASS_IDS],
}).to_string(index=False))
mixed = all_df.groupby("group")["tip"].nunique()
if (mixed > 1).any():
    print(f"\nGroups with frames of more than one class: {sorted(mixed[mixed > 1].index)}")


SEED = 42
SPLIT_FRACTIONS = {"train": 0.70, "val": 0.15, "test": 0.15}
# When a class has fewer groups than splits, its groups go to the splits in this order:
# 1 group -> train; 2 groups -> train + test (so the class is still evaluated); >= 3 -> all splits.
SUPPORT_ORDER = ("train", "test", "val")


def stratified_group_split(frame, label_col, group_col="group", fractions=SPLIT_FRACTIONS,
                           support_order=SUPPORT_ORDER, seed=SEED):
    """Assign whole groups to splits while keeping the per-class frame shares close to `fractions`.

    Groups (same creative / same source video) are atomic, so near-identical frames never
    cross splits. A group may contain frames of several classes; it is described by its
    per-class frame counts. Classes are processed from the one with the fewest groups to
    the one with the most; each carrier group (largest first, seeded order on ties) goes to
    the split with the largest remaining deficit for that class, then for total frames.
    A repair pass then guarantees every class has at least one group in the splits given by
    `support_order` (moving the smallest donor group that its own split can spare).

    Returns a dict group -> split name.
    """
    splits = list(fractions)
    counts = pd.crosstab(frame[group_col], frame[label_col])
    order = counts.index.to_numpy().copy()
    np.random.default_rng(seed).shuffle(order)
    counts = counts.loc[order]
    classes = list(counts.columns)
    sizes = counts.sum(axis=1)

    class_target = {s: counts.sum(axis=0) * f for s, f in fractions.items()}
    frame_target = {s: sizes.sum() * f for s, f in fractions.items()}
    filled = {s: pd.Series(0.0, index=classes) for s in splits}
    assignment = {}

    n_carriers = (counts > 0).sum(axis=0).sort_values(kind="stable")
    for cls in n_carriers.index:
        carriers = [g for g in counts.index if g not in assignment and counts.loc[g, cls] > 0]
        carriers.sort(key=lambda g: -counts.loc[g, cls])
        for g in carriers:
            split = max(splits, key=lambda s: (class_target[s][cls] - filled[s][cls],
                                               frame_target[s] - filled[s].sum()))
            assignment[g] = split
            filled[split] += counts.loc[g]

    def required(cls):
        return [s for s in support_order if s in splits][:min(int(n_carriers[cls]), len(splits))]

    def carriers_in(cls, split):
        return sum(1 for g, s in assignment.items() if s == split and counts.loc[g, cls] > 0)

    def can_leave(g, split):
        return all(carriers_in(c, split) > 1 or split not in required(c)
                   for c in classes if counts.loc[g, c] > 0)

    changed = True
    while changed:
        changed = False
        for cls in n_carriers.index:
            for split in required(cls):
                if carriers_in(cls, split):
                    continue
                donors = [g for g, s in assignment.items()
                          if s != split and counts.loc[g, cls] > 0 and can_leave(g, s)]
                if donors:
                    assignment[min(donors, key=lambda g: sizes[g])] = split
                    changed = True

    for cls in n_carriers.index:
        missing = [s for s in required(cls) if not carriers_in(cls, s)]
        if missing:
            print(f"  NOTE: class {cls!r} has no group in {missing} (its groups cannot be moved)")
        if n_carriers[cls] < len(splits):
            present = [s for s in splits if carriers_in(cls, s)]
            print(f"  NOTE: class {cls!r} has only {int(n_carriers[cls])} group(s); present in {present}")
    return assignment


def assert_no_group_leakage(frame, group_col="group", split_col="split"):
    splits_per_group = frame.groupby(group_col)[split_col].nunique()
    leaked = splits_per_group[splits_per_group > 1]
    assert leaked.empty, f"LEAKAGE: groups in more than one split: {list(leaked.index)}"


def print_split_summary(frame, label_col, group_col="group", split_col="split"):
    split_names = [s for s in SPLIT_FRACTIONS if s in set(frame[split_col])]
    frames = pd.crosstab(frame[label_col], frame[split_col]).reindex(columns=split_names, fill_value=0)
    groups = (frame.groupby([label_col, split_col])[group_col].nunique().unstack(fill_value=0)
              .reindex(index=frames.index, columns=split_names, fill_value=0))
    table = frames.astype(str) + " (" + groups.astype(str) + " gr)"
    total_groups = frame.groupby(label_col)[group_col].nunique().reindex(frames.index)
    table["total"] = frames.sum(axis=1).astype(str) + " (" + total_groups.astype(str) + " gr)"
    print("Frames (groups) per class and split:")
    print(table.to_string())
    totals = frames.sum(axis=0)
    print("\nFrames per split: " + ", ".join(
        f"{s}={int(totals[s])} ({totals[s] / totals.sum():.1%}, "
        f"{frame.loc[frame[split_col] == s, group_col].nunique()} groups)"
        for s in split_names))


if PILOT:
    print(f"\nPILOT MODE: keeping ~{PILOT_FRACTION:.0%} of the groups (class-balanced)")
    keep = stratified_group_split(all_df, label_col="tip",
                                  fractions={"keep": PILOT_FRACTION, "drop": 1.0 - PILOT_FRACTION},
                                  support_order=("keep",))
    all_df = all_df[all_df["group"].map(keep) == "keep"].reset_index(drop=True)
    print(f"  -> {len(all_df)} images")

print("\nSplitting...")
group_to_split = stratified_group_split(all_df, label_col="tip")
all_df["split"] = all_df["group"].map(group_to_split)
assert all_df["split"].notna().all()
assert_no_group_leakage(all_df)
print_split_summary(all_df.assign(cls=all_df["tip"].astype(str) + " " + all_df["class_name"]), label_col="cls")

train_df = all_df[all_df["split"] == "train"].reset_index(drop=True)
val_df = all_df[all_df["split"] == "val"].reset_index(drop=True)
test_df = all_df[all_df["split"] == "test"].reset_index(drop=True)
SPLIT_FRAMES = {"train": train_df, "val": val_df, "test": test_df}
Y = {split: frame["y"].to_numpy(dtype=np.int64) for split, frame in SPLIT_FRAMES.items()}
for split, frame in SPLIT_FRAMES.items():
    absent = [CLASS_NAMES[i] for i in range(NUM_CLASSES) if i not in set(Y[split])]
    if absent:
        print(f"  {split}: no images of {absent}")


def save_split_manifest(results_dir):
    """Split manifest (identical across the multi-class notebooks for the same dataset and seed)."""
    all_df[["image_name", "tip", "class_name", "group", "split"]].to_csv(Path(results_dir) / "split_manifest.csv", index=False)

## Metrics

Single-label metrics on the argmax prediction: accuracy, balanced accuracy (mean per-class recall), macro
precision/recall/F1 (averaged over the classes present in the evaluated split, so a class absent from val does not
count as F1 = 0) and support-weighted F1. **Model selection always uses validation macro-F1**, because the classes are
very imbalanced (`food_brand` alone is ~47% of the frames). The test report also shows per-class
precision/recall/F1 and the confusion matrix (rows = true class, columns = predicted class).

In [ ]:
def compute_metrics(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    present = np.unique(y_true)
    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": float(np.mean(recall_score(y_true, y_pred, labels=present, average=None, zero_division=0))),
        "macro_precision": precision_score(y_true, y_pred, labels=present, average="macro", zero_division=0),
        "macro_recall": recall_score(y_true, y_pred, labels=present, average="macro", zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, labels=present, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, labels=present, average="weighted", zero_division=0),
    }
    return {key: float(value) for key, value in metrics.items()}


METRIC_KEYS = ["accuracy", "balanced_accuracy", "macro_precision", "macro_recall", "macro_f1", "weighted_f1"]


def print_metrics(metrics, title, n_images=None, extra_lines=()):
    print("\n" + "=" * 60)
    print(title)
    print("=" * 60)
    for line in extra_lines:
        print(f"  {line}")
    if n_images is not None:
        print(f"  Images:            {n_images}")
    print(f"  Accuracy:          {metrics['accuracy']:.4f}")
    print(f"  Balanced accuracy: {metrics['balanced_accuracy']:.4f}")
    print(f"  Macro precision:   {metrics['macro_precision']:.4f}")
    print(f"  Macro recall:      {metrics['macro_recall']:.4f}")
    print(f"  Macro F1:          {metrics['macro_f1']:.4f}")
    print(f"  Weighted F1:       {metrics['weighted_f1']:.4f}")


def per_class_frame(y_true, y_pred):
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=list(range(NUM_CLASSES)), zero_division=0
    )
    return pd.DataFrame({
        "tip": CLASS_IDS, "class_name": CLASS_NAMES, "support": support,
        "precision": precision, "recall": recall, "f1": f1,
    })


def plot_confusion_matrix(y_true, y_pred, title, cmap="Blues"):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
    plt.figure(figsize=(9, 7))
    sns.heatmap(cm, annot=True, fmt="d", cmap=cmap, xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(title, fontsize=10)
    plt.xticks(rotation=45, ha="right")
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()
    return cm


def report_configuration(title, y_true, y_pred, metrics, extra_lines=(), cmap="Blues"):
    """Full test report for one configuration: metrics, per-class report and confusion matrix."""
    print_metrics(metrics, title, n_images=len(y_true), extra_lines=extra_lines)
    print("\nPER-CLASS REPORT")
    print(classification_report(y_true, y_pred, labels=list(range(NUM_CLASSES)), target_names=CLASS_NAMES,
                                digits=4, zero_division=0))
    plot_confusion_matrix(y_true, y_pred, title, cmap=cmap)
    return per_class_frame(y_true, y_pred)


def show_image(path, title):
    fig, ax = plt.subplots(figsize=(8, 5))
    with Image.open(path) as image:
        ax.imshow(image.convert("RGB"))
    ax.axis("off")
    ax.set_title(title, fontsize=9)
    plt.tight_layout()
    plt.show()


def sample_test_indices(per_class=1):
    """Seeded random test images, `per_class` of every class present in test."""
    return [idx for _, frame in test_df.groupby("y")
            for idx in frame.sample(min(per_class, len(frame)), random_state=SEED).index]


def download_results(results_dir, archive_name):
    archive_path = shutil.make_archive(str(Path.cwd() / archive_name), "zip", results_dir)
    print(f"Archive: {archive_path}")
    if IN_COLAB:
        from google.colab import files

        files.download(archive_path)


def environment_metadata():
    return {
        "python": platform.python_version(),
        "torch": torch.__version__,
        "torchvision": torchvision.__version__,
        "scikit_learn": sklearn.__version__,
        "device": str(DEVICE),
        "gpu": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else None,
        "seed": SEED,
        "pilot": PILOT,
        "classes": [{"tip": t, "name": n, "description": CLASS_DESCRIPTIONS[t]} for t, n in zip(CLASS_IDS, CLASS_NAMES)],
        "split_strategy": "group-aware stratified (whole groups per split), target 70/15/15 per class",
        "split_sizes": {split: len(frame) for split, frame in SPLIT_FRAMES.items()},
        "selection_metric": "validation macro-F1",
    }

## Prepare OpenCLIP and encode images

Validation and test images are encoded once and cached (L2-normalized), so trying many prompts is cheap.
Times exclude model setup and weight downloads.

In [ ]:
class ImagePathDataset(Dataset):
    """Loads images from disk for batched, frozen feature extraction."""

    def __init__(self, image_paths, transform):
        self.image_paths = list(image_paths)
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, index):
        with Image.open(self.image_paths[index]) as image:
            return self.transform(image.convert("RGB"))


def sync_device():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()


def image_loader(image_paths, transform, batch_size):
    return DataLoader(
        ImagePathDataset(image_paths, transform),
        batch_size=batch_size,
        shuffle=False,
        num_workers=DATALOADER_WORKERS,
        pin_memory=DEVICE.type == "cuda",
    )


import json
import shutil
import time

import open_clip

CLIP_MODEL = "ViT-B-16"
CLIP_PRETRAINED = "laion2b_s34b_b88k"
CLIP_BATCH_SIZE = 128 if DEVICE.type == "cuda" else 32

clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(CLIP_MODEL, pretrained=CLIP_PRETRAINED)
tokenizer = open_clip.get_tokenizer(CLIP_MODEL)
clip_model = clip_model.to(DEVICE).eval()
LOGIT_SCALE = float(clip_model.logit_scale.exp().item())
print(f"Loaded OpenCLIP {CLIP_MODEL} ({CLIP_PRETRAINED}) on {DEVICE}; logit scale {LOGIT_SCALE:.1f}")


@torch.inference_mode()
def encode_images(image_paths, description):
    chunks = []
    for batch in tqdm(image_loader(image_paths, clip_preprocess, CLIP_BATCH_SIZE), desc=description, leave=False):
        chunks.append(F.normalize(clip_model.encode_image(batch.to(DEVICE, non_blocking=True)).float(), dim=-1).cpu())
    return torch.cat(chunks).numpy()


image_features = {}
encode_seconds = {}
for split in ("val", "test"):
    sync_device()
    started = time.perf_counter()
    image_features[split] = encode_images(SPLIT_FRAMES[split]["image_path"].tolist(), f"Encoding {split}")
    sync_device()
    encode_seconds[split] = time.perf_counter() - started
    print(f"{split}: {image_features[split].shape} in {encode_seconds[split]:.1f}s")

## Prompts per class

The candidate concepts below are short English paraphrases of each `tip_label`. Add or edit concepts freely: selection
happens on validation only, so the test split stays untouched. A tip id that appears in the CSV without prompts raises an
error.

In [ ]:
# =====================================================================
# PROMPTS
# Every class has candidate concepts written from its tip_label meaning.
# A concept is embedded with every template below and the embeddings are
# averaged (prompt ensembling, as in the binary CLIP notebook).
# =====================================================================
TEMPLATES = [
    "a photo of {}.",
    "an image containing {}.",
    "a frame from a TV commercial showing {}.",
    "a screenshot of a television broadcast with {}.",
    "a TV frame featuring {}.",
]

CLASS_PROMPTS = {
    0: [  # Not an ad (TV programming)
        "a TV show or news program, not an advertisement",
        "a scene from a television program",
        "people talking in a TV studio",
        "a soap opera or movie scene",
    ],
    2: [  # Food or beverage products: food company / brand
        "a food or beverage brand advertisement",
        "a packaged snack or soft drink product",
        "a commercial for a cereal, candy or soda brand",
        "a close-up of a branded food package",
    ],
    3: [  # Food or beverage products promoted by a non-food company
        "a food or drink promotion by a non-food company such as a bank or phone company",
        "a telecom or bank advertisement offering food or drinks",
        "a non-food brand advertisement featuring food products",
        "a giveaway of snacks or drinks by a non-food company",
    ],
    5: [  # Supermarket / convenience store promoting food or beverages
        "a supermarket advertisement showing food products with prices",
        "a grocery store ad with fruits, vegetables and groceries",
        "a convenience store promotion of snacks and drinks",
        "a supermarket offer of discounted food",
    ],
    6: [  # Supermarket / convenience store NOT promoting food or beverages
        "a supermarket advertisement for appliances, clothes or household items",
        "a store ad showing televisions, furniture or cleaning products",
        "a convenience store advertisement without any food",
        "a retail store promotion of non-food products",
    ],
    7: [  # Restaurant / delivery / fast food promoting food or beverages
        "a fast food restaurant advertisement with burgers and fries",
        "a pizza or fried chicken delivery advertisement",
        "a restaurant meal promotion",
        "a food delivery app showing restaurant dishes",
    ],
    8: [  # Restaurant / delivery / fast food NOT promoting food or beverages
        "a restaurant brand advertisement without showing food",
        "a fast food chain ad about its stores, jobs or app, without food",
        "a restaurant logo and staff with no meals shown",
        "a food delivery service ad showing only riders and the app",
    ],
}

missing = [t for t in CLASS_IDS if not CLASS_PROMPTS.get(t)]
if missing:
    raise KeyError(f"Add prompts for these tip ids to CLASS_PROMPTS: {missing}")


_text_cache = {}


@torch.inference_mode()
def encode_concept(concept):
    """Template-ensembled, L2-normalized text embedding for one concept (cached)."""
    if concept not in _text_cache:
        tokens = tokenizer([template.format(concept) for template in TEMPLATES]).to(DEVICE)
        embeddings = F.normalize(clip_model.encode_text(tokens).float(), dim=-1)
        _text_cache[concept] = F.normalize(embeddings.mean(dim=0), dim=0)
    return _text_cache[concept]


@torch.inference_mode()
def class_embedding(tip, variant):
    """Text embedding of a class: its `variant`-th concept, or the mean of all its concepts when variant is None."""
    concepts = CLASS_PROMPTS[tip]
    if variant is None:
        return F.normalize(torch.stack([encode_concept(c) for c in concepts]).mean(dim=0), dim=0)
    return encode_concept(concepts[min(variant, len(concepts) - 1)])


@torch.inference_mode()
def class_scores(split, variant):
    """Softmax over classes of logit_scale * cos(image, class text), shape (n_images, n_classes)."""
    text = torch.stack([class_embedding(t, variant) for t in CLASS_IDS])
    logits = LOGIT_SCALE * torch.from_numpy(image_features[split]).to(DEVICE) @ text.T
    return logits.softmax(dim=-1).cpu().numpy()


N_VARIANTS = max(len(CLASS_PROMPTS[t]) for t in CLASS_IDS)
CONFIGS = {f"concept {v + 1}": v for v in range(N_VARIANTS)}
CONFIGS["ensemble (mean of all concepts)"] = None
print(f"{NUM_CLASSES} classes, {sum(len(CLASS_PROMPTS[t]) for t in CLASS_IDS)} concepts, {len(TEMPLATES)} templates, "
      f"{len(CONFIGS)} configurations")

## Prompt selection on the validation set

In [ ]:
sync_device()
started = time.perf_counter()
val_rows = []
for name, variant in CONFIGS.items():
    scores = class_scores("val", variant)
    val_rows.append({"configuration": name, **compute_metrics(Y["val"], scores.argmax(axis=1))})
sync_device()
prompt_selection_seconds = time.perf_counter() - started

# Stable sort keeps the configured order on ties; rank 1 is the primary configuration.
val_grid = pd.DataFrame(val_rows).sort_values("macro_f1", ascending=False, kind="stable").reset_index(drop=True)
val_grid.insert(0, "validation_rank", range(1, len(val_grid) + 1))
PRIMARY_CONFIG = val_grid.loc[0, "configuration"]

pd.set_option("display.width", 200)
print(f"Prompt selection: {len(CONFIGS)} configurations in {prompt_selection_seconds:.1f}s")
print("\nValidation metrics (ranked by macro-F1):")
print(val_grid[["validation_rank", "configuration"] + METRIC_KEYS].round(4).to_string(index=False))
print(f"\nPrimary configuration: {PRIMARY_CONFIG}")

## Test set evaluation

The primary result uses the configuration ranked first on validation, regardless of its test score; the others are
secondary comparisons. Every configuration gets the full test report: all metrics, the per-class report and the
confusion matrix.

In [ ]:
test_rows = []
test_outputs = {}  # configuration -> dict(scores, pred, metrics, per_class, seconds)
for rank, name in enumerate(val_grid["configuration"], 1):
    sync_device()
    started = time.perf_counter()
    scores = class_scores("test", CONFIGS[name])
    sync_device()
    scoring_seconds = time.perf_counter() - started
    pred = scores.argmax(axis=1)
    metrics = compute_metrics(Y["test"], pred)
    test_outputs[name] = {"scores": scores, "pred": pred, "metrics": metrics, "seconds": scoring_seconds}
    test_rows.append({
        "validation_rank": rank,
        "configuration": name,
        "primary": name == PRIMARY_CONFIG,
        "val_macro_f1": float(val_grid.loc[rank - 1, "macro_f1"]),
        **{f"test_{key}": value for key, value in metrics.items()},
        "test_image_encoding_seconds": encode_seconds["test"],
        "test_text_scoring_seconds": scoring_seconds,
    })

test_summary = pd.DataFrame(test_rows)
print(f"Test image encoding (shared by all configurations): {encode_seconds['test']:.2f} s")
print("CLIP training: none (pretrained weights remain frozen).\n")
print("TEST SET SUMMARY")
print(test_summary[["validation_rank", "configuration", "val_macro_f1"] + [f"test_{k}" for k in METRIC_KEYS]]
      .round(4).to_string(index=False))

# Full report for every configuration, in validation order (rank 1 = primary).
for row in test_summary.itertuples():
    out = test_outputs[row.configuration]
    out["per_class"] = report_configuration(
        f"TEST SET METRICS - CLIP {CLIP_MODEL} zero-shot, {row.configuration} - validation rank {row.validation_rank}"
        + (" (PRIMARY)" if row.primary else " (SECONDARY)"),
        Y["test"], out["pred"], out["metrics"],
        extra_lines=[f"Validation macro-F1: {row.val_macro_f1:.4f}",
                     f"Test text scoring time: {out['seconds']:.2f} s"],
        cmap="Oranges",
    )

primary = test_outputs[PRIMARY_CONFIG]

## Export results

Writes the validation grid, test summary, per-class test metrics, primary test predictions, split manifest and run metadata to `runs/clip/` (downloaded as a zip on Colab).

In [ ]:
RESULTS_DIR = Path.cwd() / "runs" / "clip"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

save_split_manifest(RESULTS_DIR)
val_grid.to_csv(RESULTS_DIR / "val_grid.csv", index=False)
test_summary.to_csv(RESULTS_DIR / "test_summary.csv", index=False)
pd.concat([out["per_class"].assign(configuration=name) for name, out in test_outputs.items()]).to_csv(
    RESULTS_DIR / "test_per_class.csv", index=False)
test_df[["image_name", "tip", "class_name", "group"]].assign(
    predicted=[CLASS_NAMES[i] for i in primary["pred"]],
    **{f"score_{name}": primary["scores"][:, i] for i, name in enumerate(CLASS_NAMES)},
).to_csv(RESULTS_DIR / "test_predictions_primary.csv", index=False)

with open(RESULTS_DIR / "run_metadata.json", "w") as f:
    json.dump({
        **environment_metadata(),
        "strategy": "clip_zero_shot",
        "clip_model": CLIP_MODEL,
        "clip_pretrained": CLIP_PRETRAINED,
        "templates": TEMPLATES,
        "class_prompts": {str(t): CLASS_PROMPTS[t] for t in CLASS_IDS},
        "primary_configuration": PRIMARY_CONFIG,
        "selection_criterion": "validation macro-F1 over prompt configurations; ties keep configured order",
        "score": "softmax over classes of logit_scale * cos(image, class text embedding)",
        "training": "none (pretrained weights frozen; train split unused)",
        "encode_seconds": encode_seconds,
        "prompt_selection_seconds": prompt_selection_seconds,
        "timing_scope": "wall-clock seconds; excludes model setup and weight downloads",
    }, f, indent=2)

print(f"Results written to {RESULTS_DIR}")
for path in sorted(RESULTS_DIR.iterdir()):
    print(f"  - {path.name}")
download_results(RESULTS_DIR, "clip_multiclass_results")

## Real images test

Qualitative predictions of the primary configuration on one seeded random test image per class. Scores are a softmax over class prompts, not calibrated probabilities.

In [ ]:
print(f"CLIP primary configuration: {PRIMARY_CONFIG}\n")
for idx in sample_test_indices(per_class=1):
    row = test_df.loc[idx]
    scores, pred = primary["scores"][idx], int(primary["pred"][idx])
    show_image(row["image_path"], f"{row['image_name']}  |  true: {row['class_name']}  |  pred: {CLASS_NAMES[pred]}")
    print(f"True: {row['class_name']} (tip {row['tip']})  Predicted: {CLASS_NAMES[pred]}  Correct: {pred == row['y']}")
    print("  Top class scores (softmax over class prompts):")
    for i in np.argsort(-scores)[:3]:
        print(f"  - {CLASS_NAMES[i]:<20}: {scores[i]:.3f}")
    print()